# Automated tomography

In [ ]:
from nqct.utils.QuantumSession import QuantumSession

qs = QuantumSession(api_key="PASTE_API_KEY_HERE", storage_path='temp/')
qs.list_backends();
#Copy over the desired backend from the given list
qs.select_backend('quantware_soprano_d2')

There is a utility to automatically generate the measurements required to perform tomography on a given state. The QASM script just has the commands required to create the state. The utility automatically adds the required headers and `qubit`/`bit` declarations and the different tomography pulses and measurement operations. The resulting measurements are stored in a bit register of size $N\cdot4^N$ for $N$ qubits, where the entries corresponding to the tomography measurements:
- The register `c` is split into groups of `N` where they are the individual qubit measurements for a given measurement
- Each group of `N` corresponds to the archetypal tomography measurements of Pauli operators. For example, with 2 qubits, the sequence would be: II, IX, IY, IZ, XI, XX, XY, XZ, YI, ..., ZY, ZZ (16 measurements in this case).

So let's take a simple example with a 2-qubit Bell state:

In [ ]:
from sqdtoolz.Utilities.DataDensityMatrix import DataDensityMatrix

my_script = r"""
h q[0];
cx q[0], q[1];
"""
my_script = DataDensityMatrix.generate_tomography_qasm(my_script, num_qubits=2, qasm_include="stdgates_transmon_fixed_coupler.inc")

qs.set_qasm(my_script)
print(my_script)

In [ ]:
qs.validate()

In [ ]:
qs.set_acquisition_type('DISCRIMINATION')
qs.set_averaging_type('SingleShotCounts')
qs.set_num_shots(1024)
qs.set_shot_repeat(1)

In [ ]:
leRes = qs.run()

In [ ]:
leRes.get_inner_slicing_vars()

In [ ]:
from sqdtoolz.Utilities.DataDensityMatrix import DataDensityMatrix
shots = leRes.get_data('c')
ddm = DataDensityMatrix.fromDataViewer(leRes, [np.linalg.inv(readout_confusion) for readout_confusion in readout_confusion_matrices])
fidelity = ddm.get_fidelity_pure_state(np.array([1,0,0,1])/np.sqrt(2))